In [ ]:
WORKER_CODE = r'''"""Worker (1 GPU): CE FOCAL vs ESTANDAR dentro de la epoca unica, con decode AGREGADO.

Reglas de banco endurecidas tras ttt05=26,39 (TTT_PLAN §17):
  - calendario EXACTO de produccion: 1 EPOCA, cada secuencia augmentada vista UNA vez
  - decode CON agregacion: 8 vistas, candidatos des-transformados, voto probmul-like, pass@2
  - 4B + eval (poblacion limpia), shardeado x4

Pregunta: la anatomia (llanura + picos de 2-10 nats en celdas de decision) dice que el CE
uniforme diluye la señal ~50x. ¿Un CE focal ((1-p)^2, que concentra el gradiente en las
celdas dificiles) mejora el resultado AGREGADO dentro del mismo presupuesto de 1 epoca?

12 tareas estratificadas por el triage: 3 mejora / 5 sobreajusta / 4 dañadas.
Prediccion prerreg: efecto modesto (+0 a +2 tareas agregadas); si el focal ayuda, deberia
verse primero en el grupo "sobreajusta" (menos gradiente gastado en celdas ya aprendidas).
"""
import glob
import json
import os
import random
import time

import numpy as np
import torch
from torch.nn.attention import SDPBackend, sdpa_kernel
from transformers import AutoModelForCausalLM

SHARD = int(os.environ.get("SHARD", "0"))
N_SHARDS = int(os.environ.get("N_SHARDS", "1"))
T00 = time.time()
_SDPA = [SDPBackend.EFFICIENT_ATTENTION, SDPBackend.FLASH_ATTENTION, SDPBackend.MATH]
IM_START, USER, ASSIST, NL, EOS = 14, 11, 12, 10, 15
ARC_TOKENS = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 15]
_TOKID = torch.tensor(ARC_TOKENS, dtype=torch.long, device="cuda:0")
THR = -float(np.log(0.2))
N_AUG = 16                     # como produccion
N_DECODE = 8                   # vistas decodificadas (las 8 primeras del set de training)
LR = 5e-5
MAX_SEQ = 3500
BUDGET_VISTA = 15

_COMP = None
for c in glob.glob("/kaggle/input/**/arc-agi_evaluation_challenges.json", recursive=True):
    _COMP = os.path.dirname(c)
    break
MDIR = None
for c in glob.glob("/kaggle/input/**/config.json", recursive=True):
    d = os.path.dirname(c)
    if glob.glob(os.path.join(d, "*.safetensors")):
        MDIR = d
        break
assert _COMP and MDIR

CHAL = json.load(open(f"{_COMP}/arc-agi_evaluation_challenges.json"))
SOLS = json.load(open(f"{_COMP}/arc-agi_evaluation_solutions.json"))

# estratificadas por el triage: 3 mejora / 5 sobreajusta / 4 dañadas
KEYS = ["38007db0", "6e453dd6", "2b83f449",
        "28a6681f", "78332cb0", "4c3d4a41", "3dc255db", "eee78d87",
        "e8686506", "7b5033c1", "b0039139", "dbff022c"]
MIS = [(gi, k) for gi, k in enumerate(KEYS) if gi % N_SHARDS == SHARD]
print(f"[shard {SHARD}] {[k for _, k in MIS]}", flush=True)


def grid_tokens(g):
    out = []
    for i, row in enumerate(g):
        out.extend(int(c) for c in row)
        if i < len(g) - 1:
            out.append(NL)
    return out


def pair_tokens_mask(inp, out):
    t = [IM_START, USER, NL] + grid_tokens(inp) + [EOS]
    m = [0] * len(t)
    head = [IM_START, ASSIST, NL]
    body = grid_tokens(out) + [EOS]
    return t + head + body, m + [0] * len(head) + [1] * len(body)


def query_tokens(inp):
    return [IM_START, USER, NL] + grid_tokens(inp) + [EOS] + [IM_START, ASSIST, NL]


def d4(g, k):
    a = np.array(g)
    if k >= 4:
        a = a.T
    for _ in range(k % 4):
        a = np.rot90(a)
    return a.tolist()


def d4_inv(g, k):
    # d4 = (transpose si k>=4) LUEGO rot90^(k%4); la inversa deshace las rots (rot90^-(k%4))
    # y despues el transpose. Verificado con 200 casos aleatorios en el builder.
    a = np.array(g)
    for _ in range(k % 4):
        a = np.rot90(a, -1)
    if k >= 4:
        a = a.T
    return a.tolist()


def aplicar(g, k, perm):
    return [[perm[c] for c in fila] for fila in d4(g, k)]


def invertir(g, k, perm):
    inv = [0] * 10
    for i, p in enumerate(perm):
        inv[p] = i
    return d4_inv([[inv[c] for c in fila] for fila in g], k)


def build_vistas(pares, seed):
    rng = random.Random(seed)
    vistas = []
    for v in range(N_AUG):
        if v == 0:
            k, perm, orden = 0, list(range(10)), list(range(len(pares)))
        else:
            k = rng.randrange(8)
            perm = list(range(10))
            rng.shuffle(perm)
            orden = list(range(len(pares)))
            rng.shuffle(orden)
        toks, mask = [], []
        for idx in orden:
            i, o = pares[idx]
            t, m = pair_tokens_mask(aplicar(i, k, perm), aplicar(o, k, perm))
            toks += t
            mask += m
        if len(toks) <= MAX_SEQ:
            vistas.append({"k": k, "perm": perm, "orden": orden, "toks": toks, "mask": mask})
    return vistas


base = AutoModelForCausalLM.from_pretrained(MDIR, torch_dtype=torch.bfloat16,
                                            device_map="cuda:0", attn_implementation="sdpa")
from peft import LoraConfig, get_peft_model, get_peft_model_state_dict, \
    set_peft_model_state_dict

LCFG = LoraConfig(r=256, lora_alpha=32, use_rslora=True, lora_dropout=0.0, bias="none",
                  task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                                  "gate_proj", "up_proj", "down_proj",
                                  "embed_tokens", "lm_head"])
MODEL = get_peft_model(base, LCFG)
for _, p_ in MODEL.named_parameters():
    if p_.requires_grad:
        p_.data = p_.data.float()
MODEL.enable_input_require_grads()
MODEL.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
INIT = {k: v.detach().clone() for k, v in get_peft_model_state_dict(MODEL).items()}
try:
    import bitsandbytes as bnb
    OPT_CLS = bnb.optim.AdamW8bit
except Exception:                                            # noqa: BLE001
    OPT_CLS = torch.optim.AdamW
print(f"[shard {SHARD}] listo | mem={torch.cuda.memory_allocated()/1e9:.1f}GB "
      f"({time.time()-T00:.0f}s)", flush=True)


@torch.no_grad()
def prefill(toks, chunk=512):
    cache, out = None, None
    for i in range(0, len(toks), chunk):
        with sdpa_kernel(_SDPA):
            out = MODEL(input_ids=torch.tensor([toks[i:i + chunk]], device="cuda:0",
                                               dtype=torch.long),
                        past_key_values=cache, use_cache=True, return_dict=True)
        cache = out.past_key_values
    return out.logits[:, -1], cache


def toks_to_grid(toks):
    rows, row = [], []
    for t in toks:
        if t == NL:
            rows.append(row)
            row = []
        elif 0 <= t <= 9:
            row.append(t)
        else:
            return None
    if row:
        rows.append(row)
    if not rows or not rows[0] or any(len(r) != len(rows[0]) for r in rows):
        return None
    return rows


@torch.no_grad()
def dfs(prompt, depth, budget=BUDGET_VISTA):
    t0 = time.time()
    last, cache0 = prefill(prompt)
    leaves, path = [], []

    def rec(logits, score, cache, left):
        if time.time() - t0 > budget or left <= 1:
            return
        lf = logits.float()
        nll = (score + torch.logsumexp(lf, dim=-1, keepdim=True)
               - lf.index_select(-1, _TOKID)).cpu()
        cands = []
        for j, t in enumerate(ARC_TOKENS):
            sc = nll[0, j].item()
            if sc < THR:
                if t == EOS:
                    leaves.append((sc, list(path)))
                else:
                    cands.append((sc, t))
        for sc, t in sorted(cands):
            if time.time() - t0 > budget:
                return
            with sdpa_kernel(_SDPA):
                o = MODEL(input_ids=torch.tensor([[t]], device="cuda:0", dtype=torch.long),
                          past_key_values=cache, use_cache=True, return_dict=True)
            path.append(t)
            rec(o.logits[:, -1], sc, o.past_key_values, left - 1)
            path.pop()

    rec(last, 0.0, cache0, depth)
    return leaves


def entrenar(vistas, focal):
    """UNA epoca: cada vista una vez, orden fijo. focal=True usa (1-p)^2 * CE por token."""
    MODEL.config.use_cache = False
    MODEL.train()
    opt = OPT_CLS([p for p in MODEL.parameters() if p.requires_grad], lr=LR)
    total = len(vistas)
    sch = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min(1.0, (s + 1) / 2) * 0.5 * (1 + np.cos(np.pi * s / max(total, 1))))
    for v in vistas:
        ids = torch.tensor([v["toks"]], device="cuda:0", dtype=torch.long)
        mask = torch.tensor(v["mask"][1:], device="cuda:0", dtype=torch.bool)
        with sdpa_kernel(_SDPA):
            out = MODEL(input_ids=ids)
        lg = out.logits[0, :-1].float()
        tgt = ids[0, 1:]
        logp = torch.log_softmax(lg, -1).gather(1, tgt.unsqueeze(1)).squeeze(1)
        nll = -logp[mask]
        if focal:
            p = torch.exp(-nll)
            loss = ((1 - p) ** 2 * nll).mean()
        else:
            loss = nll.mean()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p_ for p_ in MODEL.parameters() if p_.requires_grad], 1.0)
        opt.step()
        sch.step()
        opt.zero_grad(set_to_none=True)
    del opt, sch
    MODEL.zero_grad(set_to_none=True)
    torch.cuda.empty_cache()
    MODEL.eval()
    MODEL.config.use_cache = True


def decode_agregado(task, vistas, gt):
    """Decodifica N_DECODE vistas, des-transforma candidatos y vota probmul-like."""
    votos = {}
    gt_en_alguna = False
    for v in vistas[:N_DECODE]:
        prompt = []
        for idx in v["orden"]:
            p_ = task["train"][idx]
            t, _ = pair_tokens_mask(aplicar(p_["input"], v["k"], v["perm"]),
                                    aplicar(p_["output"], v["k"], v["perm"]))
            prompt += t
        prompt += query_tokens(aplicar(task["test"][0]["input"], v["k"], v["perm"]))
        gt_v = grid_tokens(aplicar(gt, v["k"], v["perm"])) + [EOS]
        hojas = dfs(prompt, min(931, len(gt_v) + 40))
        for sc, camino in hojas:
            g = toks_to_grid(camino)
            if g is None:
                continue
            try:
                g0 = invertir(g, v["k"], v["perm"])
            except Exception:                                # noqa: BLE001
                continue
            h = json.dumps(g0)
            votos[h] = votos.get(h, 0.0) + max(0.0, 3.0 - sc)
            if g0 == gt:
                gt_en_alguna = True
    orden = sorted(votos.items(), key=lambda x: -x[1])
    top2 = [json.loads(h) for h, _ in orden[:2]]
    return {"resuelta": bool(any(g == gt for g in top2)),
            "gt_en_pool": gt_en_alguna, "n_candidatos": len(votos),
            "votos_top": [round(s, 3) for _, s in orden[:3]]}


RES = {"tareas": []}
for gi, k in MIS:
    task = CHAL[k]
    gt = SOLS[k][0]
    pares = [(p["input"], p["output"]) for p in task["train"]]
    vistas = build_vistas(pares, seed=5000 + gi)
    fila = {"task": k, "n_vistas": len(vistas)}
    for arm, focal in (("estandar", False), ("focal", True)):
        set_peft_model_state_dict(MODEL, {kk: v.clone() for kk, v in INIT.items()})
        t0 = time.time()
        entrenar(vistas, focal)
        t_train = time.time() - t0
        r = decode_agregado(task, vistas, gt)
        r["train_s"] = round(t_train, 1)
        fila[arm] = r
        torch.cuda.empty_cache()
    print(f"[shard {SHARD}] {k}: estandar resuelta={fila['estandar']['resuelta']} "
          f"gt_pool={fila['estandar']['gt_en_pool']} | focal resuelta={fila['focal']['resuelta']} "
          f"gt_pool={fila['focal']['gt_en_pool']} ({time.time()-T00:.0f}s)", flush=True)
    RES["tareas"].append(fila)
    json.dump(RES, open(f"/kaggle/working/shard{SHARD}.json", "w"))

json.dump(RES, open(f"/kaggle/working/shard{SHARD}.json", "w"))
print(f"[shard {SHARD}] fin {time.time()-T00:.0f}s", flush=True)
'''

import json
import os
import subprocess
import sys
import time

open("/kaggle/working/worker.py", "w").write(WORKER_CODE)
N_GPUS = 4
T0 = time.time()
procs = []
for i in range(N_GPUS):
    env = dict(os.environ, SHARD=str(i), N_SHARDS=str(N_GPUS),
               CUDA_VISIBLE_DEVICES=str(i), PYTHONUNBUFFERED="1")
    p = subprocess.Popen([sys.executable, "/kaggle/working/worker.py"],
                         stdout=open(f"/kaggle/working/shard{i}.log", "w"),
                         stderr=subprocess.STDOUT, env=env)
    procs.append(p)
for i, p in enumerate(procs):
    rc = p.wait()
    print(f"[main] worker {i} rc={rc} ({time.time()-T0:.0f}s)", flush=True)
    if rc != 0:
        print(open(f"/kaggle/working/shard{i}.log", errors="replace").read()[-3000:],
              flush=True)
tareas = []
for i in range(N_GPUS):
    f = f"/kaggle/working/shard{i}.json"
    if os.path.exists(f):
        tareas += json.load(open(f))["tareas"]
json.dump({"tareas": tareas}, open("/kaggle/working/ttt_focal.json", "w"))
res = {"n": len(tareas),
       "estandar_resueltas": sum(t["estandar"]["resuelta"] for t in tareas),
       "focal_resueltas": sum(t["focal"]["resuelta"] for t in tareas),
       "estandar_gt_pool": sum(t["estandar"]["gt_en_pool"] for t in tareas),
       "focal_gt_pool": sum(t["focal"]["gt_en_pool"] for t in tareas),
       "wall_s": round(time.time() - T0, 1)}
print("RESULT_JSON:" + json.dumps(res), flush=True)
